# Week 3 — Font Recommendation Engine
**Tools:** OpenCV, scikit-learn (KNN), NumPy
**Dataset:** Font Dataset (Kaggle) — class-labelled image folders

In [ ]:
!pip install scikit-learn opencv-python-headless matplotlib numpy pandas joblib -q
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt,json,joblib
from pathlib import Path
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score
print('Libraries loaded ✓')

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
FONT_PATH='/content/drive/MyDrive/BrandSphere/font_dataset/'
def load_fonts(base,img_size=32):
    imgs,labels,classes=[],[],[]
    base=Path(base); cls_dirs=sorted([d for d in base.iterdir() if d.is_dir()])
    classes=[d.name for d in cls_dirs]; c2i={c:i for i,c in enumerate(classes)}
    for cls_dir in cls_dirs:
        for f in list(cls_dir.glob('*.jpg'))+list(cls_dir.glob('*.png')):
            try:
                img=cv2.imread(str(f),cv2.IMREAD_GRAYSCALE)
                img=cv2.resize(img,(img_size,img_size)); imgs.append(img.flatten()/255.0); labels.append(c2i[cls_dir.name])
            except: pass
    print(f'Font classes: {classes}'); print(f'Images loaded: {len(imgs)}')
    return np.array(imgs),np.array(labels),classes
X_f,y_f,font_classes=load_fonts(FONT_PATH)
X_tr,X_te,y_tr,y_te=train_test_split(X_f,y_f,test_size=.2,stratify=y_f,random_state=42)

In [ ]:
knn=KNeighborsClassifier(n_neighbors=5,metric='euclidean')
knn.fit(X_tr,y_tr); y_pred=knn.predict(X_te)
print(f'KNN Accuracy: {accuracy_score(y_te,y_pred):.4f}')
print(classification_report(y_te,y_pred,target_names=font_classes))
joblib.dump(knn,'font_knn.pkl'); print('Model saved ✓')

In [ ]:
# Font-to-brand personality mapping
FONT_MAP={
    'serif':      {'tones':['Luxury & Premium','Professional & Corporate'],'industries':['Finance','Law','Real Estate'],'examples':['Georgia','Playfair Display'],'reason':'Trust, tradition, authority'},
    'sans_serif': {'tones':['Minimalist','Tech-Forward & Innovative'],'industries':['Technology','Healthcare','E-Commerce'],'examples':['Inter','Helvetica Neue','Space Grotesk'],'reason':'Clarity, modernity, approachability'},
    'script':     {'tones':['Youthful & Playful'],'industries':['Fashion & Apparel','Food & Beverage'],'examples':['Pacifico','Dancing Script'],'reason':'Elegance, creativity, personality'},
    'display':    {'tones':['Bold & Vibrant'],'industries':['Entertainment','Sports','Gaming'],'examples':['Bebas Neue','Anton'],'reason':'Power, excitement, high energy'},
    'monospace':  {'tones':['Tech-Forward & Innovative'],'industries':['Technology','Cybersecurity'],'examples':['Space Mono','Courier'],'reason':'Precision, technical expertise'},
}
def recommend_font(industry,tone):
    recs=[]
    for fam,d in FONT_MAP.items():
        score=0
        if industry in d['industries']: score+=2
        if any(t in tone for t in d['tones']): score+=2
        recs.append((score,fam,d))
    return sorted(recs,reverse=True)
print('NovaTech (Technology / Tech-Forward) font recommendations:')
for score,fam,d in recommend_font('Technology','Tech-Forward & Innovative'):
    print(f'  {fam} (score={score}): {d["examples"][0]} — {d["reason"]}')

In [ ]:
with open('font_map.json','w') as f: json.dump(FONT_MAP,f,indent=2)
print('Font map saved ✓')

## ✅ Week 3 Deliverables
- [ ] Font images loaded from Kaggle dataset (grayscale, 32×32)
- [ ] KNN classifier trained and saved (font_knn.pkl)
- [ ] Font-to-brand personality mapping table built
- [ ] Recommendation function: industry + tone → font family + examples